# Hyperparameter Tuning: Grid Search and Randomized Search

This notebook is for **classification**. Update `DATA_PATH` and `TARGET_COLUMN` below before running all cells. It compares Logistic Regression, Random Forest, and SVC with GridSearchCV and RandomizedSearchCV. The test set is held out until final evaluation.

In [ ]:
from pathlib import Path
import json, time
import numpy as np
import pandas as pd
import sklearn
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV, RandomizedSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.svm import SVC

RANDOM_STATE = 42
DATA_PATH = Path("../data/your_dataset.csv")  # change to your CSV path
TARGET_COLUMN = "target"                     # change to your label column
OUTPUT_DIR = Path("../results")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("Scikit-learn:", sklearn.__version__)

## 1. Load and inspect the dataset

In [ ]:
if not DATA_PATH.exists():
    raise FileNotFoundError(f"Put your CSV at {DATA_PATH} or update DATA_PATH.")
df = pd.read_csv(DATA_PATH)
print("Shape:", df.shape)
display(df.head())
print(df.dtypes)
print("Missing values per column:\n", df.isna().sum())
if TARGET_COLUMN not in df.columns:
    raise ValueError(f"Set TARGET_COLUMN to one of: {list(df.columns)}")
df = df.dropna(subset=[TARGET_COLUMN]).copy()
X = df.drop(columns=[TARGET_COLUMN])
y = df[TARGET_COLUMN]
print("Class counts:\n", y.value_counts(dropna=False))

## 2. Split data and build leakage-safe preprocessing

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)
numeric = X_train.select_dtypes(include=["number", "bool"]).columns.tolist()
categorical = [c for c in X_train.columns if c not in numeric]
numeric_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])
categorical_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])
preprocessor = ColumnTransformer([
    ("numeric", numeric_pipe, numeric),
    ("categorical", categorical_pipe, categorical),
])
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
print("Train rows:", len(X_train), "| Test rows:", len(X_test))

## 3. Define three algorithms and their search spaces

In [ ]:
models = {
    "LogisticRegression": (
        LogisticRegression(max_iter=3000, random_state=RANDOM_STATE),
        {"model__C": [0.01, 0.1, 1.0, 10.0],
         "model__solver": ["lbfgs"],
         "model__class_weight": [None, "balanced"]},
    ),
    "RandomForest": (
        RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=1),
        {"model__n_estimators": [100, 200, 300],
         "model__max_depth": [None, 10, 20],
         "model__min_samples_split": [2, 5, 10],
         "model__min_samples_leaf": [1, 2, 4],
         "model__class_weight": [None, "balanced"]},
    ),
    "SVC": (
        SVC(),
        {"model__C": [0.1, 1.0, 10.0, 100.0],
         "model__kernel": ["linear", "rbf"],
         "model__gamma": ["scale", "auto"],
         "model__class_weight": [None, "balanced"]},
    ),
}
SCORING = "f1_weighted"
RANDOM_SEARCH_ITERATIONS = 12
print("Models:", list(models))

## 4. Run GridSearchCV and RandomizedSearchCV

In [ ]:
results = []
best_estimators = {}
best_params_summary = {}

for model_name, (estimator, param_grid) in models.items():
    pipe = Pipeline([("preprocess", preprocessor), ("model", estimator)])

    for search_name in ["GridSearchCV", "RandomizedSearchCV"]:
        if search_name == "GridSearchCV":
            search = GridSearchCV(pipe, param_grid=param_grid, scoring=SCORING,
                                  cv=cv, n_jobs=-1, refit=True, error_score="raise")
        else:
            search = RandomizedSearchCV(pipe, param_distributions=param_grid,
                                        n_iter=RANDOM_SEARCH_ITERATIONS, scoring=SCORING,
                                        cv=cv, n_jobs=-1, random_state=RANDOM_STATE,
                                        refit=True, error_score="raise")
        start = time.perf_counter()
        search.fit(X_train, y_train)
        elapsed = time.perf_counter() - start
        best_estimators[(model_name, search_name)] = search.best_estimator_
        best_params_summary[f"{model_name}_{'grid' if search_name == 'GridSearchCV' else 'randomized'}"] = {
            "best_params": search.best_params_,
            "best_cv_score": float(search.best_score_),
            "search_seconds": elapsed,
        }
        y_pred = search.best_estimator_.predict(X_test)
        results.append({
            "model": model_name, "search": search_name,
            "best_cv_f1_weighted": search.best_score_,
            "test_accuracy": accuracy_score(y_test, y_pred),
            "test_precision_weighted": precision_score(y_test, y_pred, average="weighted", zero_division=0),
            "test_recall_weighted": recall_score(y_test, y_pred, average="weighted", zero_division=0),
            "test_f1_weighted": f1_score(y_test, y_pred, average="weighted", zero_division=0),
            "search_seconds": elapsed,
            "best_params": json.dumps(search.best_params_, default=str),
        })
        print(model_name, search_name, "best CV weighted F1:", round(search.best_score_, 4),
              "| seconds:", round(elapsed, 2))

results_df = pd.DataFrame(results).sort_values(
    ["test_f1_weighted", "search_seconds"], ascending=[False, True]
).reset_index(drop=True)
display(results_df)

## 5. Save results and inspect the selected model

In [ ]:
results_df.to_csv(OUTPUT_DIR / "model_comparison.csv", index=False)
with (OUTPUT_DIR / "best_parameters.json").open("w", encoding="utf-8") as f:
    json.dump(best_params_summary, f, indent=2, default=str)

winner = results_df.iloc[0]
winner_key = (winner["model"], winner["search"])
best_model = best_estimators[winner_key]
joblib_path = OUTPUT_DIR / "best_model.joblib"
import joblib
joblib.dump(best_model, joblib_path)
winner_pred = best_model.predict(X_test)
print("Selected:", winner["model"], "-", winner["search"])
print("Accuracy:", round(accuracy_score(y_test, winner_pred), 4))
print("Weighted F1:", round(f1_score(y_test, winner_pred, average="weighted", zero_division=0), 4))
print(classification_report(y_test, winner_pred, zero_division=0))
print("Saved comparison and parameters in", OUTPUT_DIR.resolve())

## 6. Discussion for your report

Use the measured values in `results/model_comparison.csv`; do not fill in invented numbers.

- Which model has the strongest test weighted F1?
- Did GridSearchCV or RandomizedSearchCV reach a better score for each algorithm?
- How much search time did each method take?
- Is the performance difference meaningful for your application?
- Discuss interpretability, model size, class-wise errors, and limitations.

Copy your real results into `docs/comparative_analysis.md` before submitting.